## VideoFake-HeadSwap | Geniş kafa / saç çizgisi swap (FaceFusion)

Klasik VideoFake sadece yüzü değiştirir. Bu notebook FaceFusion ile maskeyi **saç çizgisine doğru** genişletir (alın + üst kafa).

**Beklenti:** Saç modelini sihirli şekilde değiştirmez. En iyi sonuç: kaynak foto ile videodaki saç benzer olsun + `HairBias` yüksek.

[Open in Colab](https://colab.research.google.com/github/ixelizm/VideoFake-HeadSwap/blob/main/HeadSwap_colab.ipynb)

Runtime → **GPU** zorunlu.

In [ ]:
#@title 0. GPU kontrol
import sys, subprocess
print("Python:", sys.version)
subprocess.run(["nvidia-smi"], check=False)
!ffmpeg -version | head -n 1

In [ ]:
#@title 1. FaceFusion kurulum
import os, sys, subprocess

FF_DIR = "/content/facefusion"
if not os.path.isdir(FF_DIR):
    !git clone --depth 1 https://github.com/facefusion/facefusion.git {FF_DIR}
else:
    %cd {FF_DIR}
    !git pull --ff-only || true

%cd {FF_DIR}
# Colab: CUDA 12/13 — install.py cuda@12 genelde T4 ile uyumlu
!python install.py cuda@12 --skip-conda
print("Install done. cwd=", os.getcwd())

In [ ]:
#@title 2. Kaynak yüz + hedef video yükle
import os
from google.colab import files

UPLOAD_DIR = "/content/headswap_uploads"
os.makedirs(UPLOAD_DIR, exist_ok=True)
output = "/content/headswap_out.mp4"

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
VIDEO_EXTS = {".mp4", ".mov", ".avi", ".mkv", ".webm"}

def _save_upload(prompt, allowed_exts, default_name):
    print(prompt)
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("Dosya seçilmedi — hücreyi tekrar çalıştır.")
    name = next(iter(uploaded))
    ext = os.path.splitext(name)[1].lower()
    if ext not in allowed_exts:
        raise RuntimeError(f"Geçersiz: {name}. İzin: {sorted(allowed_exts)}")
    dest = os.path.join(UPLOAD_DIR, default_name + ext)
    with open(dest, "wb") as f:
        f.write(uploaded[name])
    print(f"OK: {dest} ({len(uploaded[name])/1024/1024:.2f} MB)")
    return dest

source = _save_upload("1/2 — KAYNAK YÜZ (jpg/png):", IMAGE_EXTS, "face")
target = _save_upload("2/2 — HEDEF VİDEO (mp4/...):", VIDEO_EXTS, "video")
print("\nsource:", source)
print("target:", target)
print("output:", output)

In [ ]:
#@title 3. Head-biased swap çalıştır
import os, sys, shlex, subprocess

source = globals().get("source")
target = globals().get("target")
output = globals().get("output", "/content/headswap_out.mp4")
assert source and os.path.isfile(source), "Önce hücre 2 ile yüz yükle"
assert target and os.path.isfile(target), "Önce hücre 2 ile video yükle"

HairBias = 35 #@param {type:"slider", min:0, max:80, step:5}
#@markdown HairBias = üst (saç) padding. Yüksek = daha fazla alın/saç çizgisi, artefakt riski artar.
SidePadding = 15 #@param {type:"slider", min:0, max:40, step:5}
BottomPadding = 10 #@param {type:"slider", min:0, max:40, step:5}
MaskBlur = 0.4 #@param {type:"slider", min:0.0, max:1.0, step:0.05}
UseEnhancer = True #@param {type:"boolean"}
ExecutionProviders = "cuda" #@param ["cuda", "cpu"]

processors = ["face_swapper"]
if UseEnhancer:
    processors.append("face_enhancer")

# top right bottom left — top = HairBias
padding = f"{HairBias} {SidePadding} {BottomPadding} {SidePadding}"

args = [
    sys.executable, "facefusion.py", "headless-run",
    "--source-paths", source,
    "--target-path", target,
    "--output-path", output,
    "--processors", *processors,
    "--face-mask-types", "box", "occlusion",
    "--face-mask-padding", *padding.split(),
    "--face-mask-blur", str(MaskBlur),
    "--execution-providers", ExecutionProviders,
    "--output-video-quality", "18",
    "--output-video-encoder", "libx264",
]

print("padding top/right/bottom/left:", padding)
print("cmd:", " ".join(shlex.quote(a) for a in args))
proc = subprocess.run(args, cwd="/content/facefusion")
print("exit:", proc.returncode)

if os.path.isfile(output) and os.path.getsize(output) > 0:
    print(f"OUTPUT OK: {output} ({os.path.getsize(output)/1024/1024:.2f} MB)")
else:
    raise RuntimeError("Çıktı oluşmadı — yukarıdaki FaceFusion loguna bak. Gerekirse ExecutionProviders=cpu dene.")

In [ ]:
#@title 4. Sonucu indir
import os
from google.colab import files

output = globals().get("output", "/content/headswap_out.mp4")
if os.path.isfile(output) and os.path.getsize(output) > 0:
    print("İndiriliyor:", output, round(os.path.getsize(output)/1024/1024, 2), "MB")
    files.download(output)
else:
    print("Dosya yok — hücre 3’ü tekrar çalıştır.")